# 01 - Fixed-fleet smart-meter producer

Publishes exactly 10,000 detailed v2 events for the current 15-minute interval. Schedule this notebook every 15 minutes for two days.

In [ ]:
import hashlib, json, math, random, time, uuid
from base64 import b64encode
from datetime import datetime, timedelta, timezone
from pathlib import Path
import oci

FLEET_SIZE, BATCH_SIZE, NAMESPACE = 10_000, 500, uuid.UUID('7ea59664-2b40-4b4d-a739-29406c3c25ae')
config = {}
for raw in Path('/Workspace/ORACLE_AIDP_STREAMING/config/runtime.yaml').read_text(encoding='utf-8').splitlines():
    line = raw.strip()
    if line and not line.startswith('#') and ':' in line:
        key, value = line.split(':', 1); config[key.strip()] = value.strip().strip(chr(34)).strip(chr(39))
required = ('OCI_STREAM_ENDPOINT', 'OCI_STREAM_OCID', 'OCI_CONFIG_FILE', 'OCI_PROFILE')
missing = [key for key in required if not config.get(key)]
if missing: raise ValueError('runtime.yaml is missing: ' + ', '.join(missing))
if config.get('METER_COUNT') and int(config['METER_COUNT']) != FLEET_SIZE:
    raise ValueError('METER_COUNT is fixed at 10000 for this production pipeline; set it to 10000 or remove it.')

def interval_start(value):
    instant = datetime.fromisoformat(value.replace('Z', '+00:00')) if value else datetime.now(timezone.utc)
    instant = instant.astimezone(timezone.utc).replace(second=0, microsecond=0)
    return instant - timedelta(minutes=instant.minute % 15)

def build_event(number, start):
    meter_id = f'MTR-{number:06d}'
    seed = int(hashlib.sha256(f'v2|{meter_id}|{start.isoformat()}'.encode()).hexdigest()[:16], 16)
    rng = random.Random(seed)
    region = f'REG-{((number - 1) % 20) + 1:02d}'; feeder = f'FDR-{((number - 1) % 200) + 1:03d}'; transformer = f'TRF-{((number - 1) % 1000) + 1:04d}'
    segment = ('RESIDENTIAL', 'SMALL_BUSINESS', 'INDUSTRIAL')[number % 3]
    temp = 28 + 7 * math.sin((start.hour - 13) * math.pi / 12) + rng.uniform(-1.5, 1.5)
    humidity = min(95, max(20, 63 - 1.2 * (temp - 28) + rng.gauss(0, 4)))
    profile = {'RESIDENTIAL': 0.45, 'SMALL_BUSINESS': 1.2, 'INDUSTRIAL': 3.0}[segment]
    time_load = 0.35 + 0.85 * max(0, math.sin((start.hour - 8) * math.pi / 12))
    weekend = 0.88 if start.weekday() >= 5 else 1.0
    kwh = round(max(0.02, profile * time_load * weekend * (1 + .012 * max(0, temp - 30)) + rng.gauss(0, profile * .035)), 4)
    voltage = round(230 + rng.gauss(0, 2.5), 2); pf = round(min(1, max(.78, .965 + rng.gauss(0, .012))), 3)
    demand = round(max(.05, kwh * 4 * rng.uniform(.94, 1.12)), 4)
    tariff = 'PEAK' if 17 <= start.hour < 22 else 'SHOULDER' if 7 <= start.hour < 17 else 'OFF_PEAK'
    outage = 15 if rng.random() < 0.0002 else 0; tamper = rng.random() < 0.0001
    device_events = (['OUTAGE_RESTORED'] if outage else []) + (['TAMPER_ALERT'] if tamper else [])
    return {'schema_version':'2.0','event_id':str(uuid.uuid5(NAMESPACE, f'{meter_id}|{start.isoformat()}')),'event_type':'INTERVAL_READING','event_ts_utc':(start + timedelta(minutes=15, seconds=rng.randint(1, 90))).isoformat(),'meter_id':meter_id,'device_id':f'DEV-{number:06d}','service_point_id':f'SP-{number:06d}','service_point_type':'ELECTRIC','interval_start_utc':start.isoformat(),'interval_end_utc':(start + timedelta(minutes=15)).isoformat(),'interval_minutes':15,'consumption_kwh':kwh,'demand_kw':demand,'voltage_v':voltage,'current_a':round(demand * 1000 / voltage / pf, 3),'power_factor':pf,'frequency_hz':round(50 + rng.gauss(0, .03), 3),'temperature_c':round(temp, 2),'humidity_pct':round(humidity, 2),'quality_code':'ACTUAL','tariff_code':tariff,'meter_status':'ACTIVE','firmware_version':f'2.{(number % 4) + 1}.0','region_code':region,'feeder_id':feeder,'transformer_id':transformer,'customer_segment':segment,'outage_minutes':outage,'tamper_flag':tamper,'measurement_events':[],'device_events':device_events}

start = interval_start(config.get('INTERVAL_START_UTC', ''))
events = [build_event(number, start) for number in range(1, FLEET_SIZE + 1)]
client = oci.streaming.StreamClient(oci.config.from_file(config['OCI_CONFIG_FILE'], config['OCI_PROFILE']), service_endpoint=config['OCI_STREAM_ENDPOINT'])
for batch_start in range(0, FLEET_SIZE, BATCH_SIZE):
    batch = events[batch_start:batch_start + BATCH_SIZE]
    entries = [oci.streaming.models.PutMessagesDetailsEntry(key=b64encode(row['meter_id'].encode()).decode(), value=b64encode(json.dumps(row, separators=(',', ':')).encode()).decode()) for row in batch]
    response = client.put_messages(config['OCI_STREAM_OCID'], oci.streaming.models.PutMessagesDetails(messages=entries)).data.entries
    errors = [entry.error_message for entry in response if entry.error]
    if errors: raise RuntimeError(f'OCI rejected batch {batch_start // BATCH_SIZE + 1}: {errors[:3]}')
print(f'Published {FLEET_SIZE:,} unique meter events for {start.isoformat()} in {FLEET_SIZE // BATCH_SIZE} batches.')